# 🎮 Projeto Final — League OF Legends Personagens
<img src = "https://tse4.mm.bing.net/th/id/OIP.MgIJGb8JEg0-DmQy2qJrRgHaEK?rs=1&pid=ImgDetMain&o=7&rm=3">


## ✅ Checklist de Entrega
- [x] Descrição do problema e público-alvo
- [x] Carregamento do dataset `games_catalog_G4.csv`
- [ ] Dicionário de dados
- [ ] EDA (exploração, gráficos, estatísticas)
- [ ] Limpeza (ausentes/outliers) e justificativas
- [ ] Vetorização de gêneros (multi-hot)
- [ ] Similaridade cosseno
- [ ] Função de recomendação + casos de teste
- [ ] Insights e próximos passos


## 1. Descrição do Problema
- Qual problema você quer resolver?
- Quem é o usuário final?
- Qual é o critério de sucesso (ex.: **recomendar 5 jogos similares** a um título dado)?

**Escreva aqui:**

## 2. Carregando o Dataset
Use o arquivo `https://raw.githubusercontent.com/fisicorj/aulacienciadedados/refs/heads/main/games_catalog_G4.csv`. Se estiver no Colab, faça upload ou monte o Drive.

Carregue as blibiotecas necessárias

In [80]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [2]:
URL_RAW = "https://raw.githubusercontent.com/MatheusJosePereira/League-of-Legends-Similaridade-Cosseno/refs/heads/v1/Dados/LoL_champions.csv"

In [30]:
df = pd.read_csv(URL_RAW)

,Name,Tags,Role,Range type,Resourse type,Base HP,HP per lvl,Base mana,Mana per lvl,Movement speed,...,Attack range,HP regeneration,HP regeneration per lvl,Mana regeneration,Mana regeneration per lvl,Attack damage,Attack damage per lvl,Attack speed per lvl,Attack speed,AS ratio
0,Aatrox,Fighter,Top,Melee,Blood Well,650,114,0,0.0,345,...,175,3.00,0.50,0.0,0.0,60,5.00,2.500,0.651,0.651
1,Ahri,"Mage,Assassin",Middle,Ranged,Mana,590,104,418,25.0,330,...,550,2.50,0.60,8.0,0.8,53,3.00,2.200,0.668,0.625
2,Akali,Assassin,"Top,Middle",Melee,Energy,600,119,200,0.0,345,...,125,9.00,0.90,50.0,0.0,62,3.30,3.200,0.625,0.625
3,Akshan,"Marksman,Assassin",Middle,Ranged,Mana,630,107,350,40.0,330,...,500,3.75,0.65,8.2,0.7,52,3.00,4.000,0.638,0.400
4,Alistar,"Tank,Support",Support,Melee,Mana,685,120,350,40.0,330,...,125,8.50,0.85,8.5,0.8,62,3.75,2.125,0.625,0.625


## 4. EDA — Análise Exploratória

### 4.1 Estrutura e tipos

In [7]:
df.head()

,Name,Tags,Role,Range type,Resourse type,Base HP,HP per lvl,Base mana,Mana per lvl,Movement speed,...,Attack range,HP regeneration,HP regeneration per lvl,Mana regeneration,Mana regeneration per lvl,Attack damage,Attack damage per lvl,Attack speed per lvl,Attack speed,AS ratio
0,Aatrox,Fighter,Top,Melee,Blood Well,650,114,0,0.0,345,...,175,3.00,0.50,0.0,0.0,60,5.00,2.500,0.651,0.651
1,Ahri,"Mage,Assassin",Middle,Ranged,Mana,590,104,418,25.0,330,...,550,2.50,0.60,8.0,0.8,53,3.00,2.200,0.668,0.625
2,Akali,Assassin,"Top,Middle",Melee,Energy,600,119,200,0.0,345,...,125,9.00,0.90,50.0,0.0,62,3.30,3.200,0.625,0.625
3,Akshan,"Marksman,Assassin",Middle,Ranged,Mana,630,107,350,40.0,330,...,500,3.75,0.65,8.2,0.7,52,3.00,4.000,0.638,0.400
4,Alistar,"Tank,Support",Support,Melee,Mana,685,120,350,40.0,330,...,125,8.50,0.85,8.5,0.8,62,3.75,2.125,0.625,0.625


In [11]:
df[['Name', 'Tags', 'Role', 'Range type']].dtypes

,0
Name,object
Tags,object
Role,object
Range type,object


In [13]:
df.isnull().sum()

,0
Name,0
Tags,0
Role,0
Range type,0
Resourse type,7
Base HP,0
HP per lvl,0
Base mana,0
Mana per lvl,0
Movement speed,0


In [15]:
df_null = df[df['Resourse type'].isnull()]

In [16]:
df_null.head(7)

,Name,Tags,Role,Range type,Resourse type,Base HP,HP per lvl,Base mana,Mana per lvl,Movement speed,...,Attack range,HP regeneration,HP regeneration per lvl,Mana regeneration,Mana regeneration per lvl,Attack damage,Attack damage per lvl,Attack speed per lvl,Attack speed,AS ratio
13,Bel'Veth,Fighter,Jungle,Melee,NaN,610,99,60,0.0,340,...,175,6.0,0.6,0.0,0.0,60,1.5,0.00,0.850,0.850
26,Dr. Mundo,"Tank,Fighter",Top,Melee,NaN,613,103,0,0.0,345,...,125,7.0,0.5,0.0,0.0,61,2.5,3.30,0.670,0.625
36,Garen,"Fighter,Tank",Top,Melee,NaN,690,98,0,0.0,340,...,175,8.0,0.5,0.0,0.0,69,4.5,3.65,0.625,0.625
58,Katarina,"Assassin,Mage",Middle,Melee,NaN,672,108,0,0.0,335,...,125,7.5,0.7,0.0,0.0,58,3.2,2.74,0.658,0.658
107,Riven,"Fighter,Assassin",Top,Melee,NaN,630,100,0,0.0,340,...,125,8.5,0.5,0.0,0.0,64,3.0,3.50,0.625,0.625
147,Viego,"Fighter,Assassin",Jungle,Melee,NaN,630,109,0,0.0,345,...,200,7.0,0.7,0.0,0.0,57,3.5,2.50,0.658,0.658
160,Zac,"Tank,Fighter","Jungle,Top,Support",Melee,NaN,685,109,0,0.0,340,...,175,5.0,0.5,0.0,0.0,60,3.4,1.60,0.736,0.638


In [58]:
df['Resourse type'].fillna('Nenhum', inplace=True)

In [59]:
df['Resourse type'].unique()

array(['Blood Well', 'Mana', 'Energy', 'Nenhum', 'Fury', 'Rage',
       'Courage', 'Shield', 'Ferocity', 'Heat', 'Grit', 'Crimson Rush',
       'Flow'], dtype=object)

##Criando dataset Similaridade Cosseno

In [66]:
df_binario = pd.get_dummies(df[['Resourse type', 'Range type']])
df_binario = df_binario.astype(int)
df_binario.head()

,Resourse type_Blood Well,Resourse type_Courage,Resourse type_Crimson Rush,Resourse type_Energy,Resourse type_Ferocity,Resourse type_Flow,Resourse type_Fury,Resourse type_Grit,Resourse type_Heat,Resourse type_Mana,Resourse type_Nenhum,Resourse type_Rage,Resourse type_Shield,Range type_Melee,Range type_Ranged
0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1
2,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0


In [70]:
df_v1 = pd.concat([df['Name'], df_binario], axis=1)

In [71]:
df_v1.head()

,Name,Resourse type_Blood Well,Resourse type_Courage,Resourse type_Crimson Rush,Resourse type_Energy,Resourse type_Ferocity,Resourse type_Flow,Resourse type_Fury,Resourse type_Grit,Resourse type_Heat,Resourse type_Mana,Resourse type_Nenhum,Resourse type_Rage,Resourse type_Shield,Range type_Melee,Range type_Ranged
0,Aatrox,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,Ahri,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1
2,Akali,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0
3,Akshan,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1
4,Alistar,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0


In [72]:
df_rotas = df['Role'].str.get_dummies(sep=',')
df_rotas.head()

,Bottom,Jungle,Middle,Support,Top
0,0,0,0,0,1
1,0,0,1,0,0
2,0,0,1,0,1
3,0,0,1,0,0
4,0,0,0,1,0


In [73]:
df_v2 = pd.concat([df_v1, df_rotas], axis=1)
df_v2.head()

,Name,Resourse type_Blood Well,Resourse type_Courage,Resourse type_Crimson Rush,Resourse type_Energy,Resourse type_Ferocity,Resourse type_Flow,Resourse type_Fury,Resourse type_Grit,Resourse type_Heat,...,Resourse type_Nenhum,Resourse type_Rage,Resourse type_Shield,Range type_Melee,Range type_Ranged,Bottom,Jungle,Middle,Support,Top
0,Aatrox,1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
1,Ahri,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
2,Akali,0,0,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,1
3,Akshan,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,Alistar,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0


In [75]:
valor = df_v2.drop('Name', axis=1).values
simcos = cosine_similarity(valor)

In [77]:
df_sim = pd.DataFrame(simcos, columns=df_v2['Name'], index=df_v2['Name'])
df_sim.head()

Name,Aatrox,Ahri,Akali,Akshan,Alistar,Amumu,Anivia,Annie,Aphelios,Ashe,...,Yone,Yorick,Yuumi,Zac,Zed,Zeri,Ziggs,Zilean,Zoe,Zyra
Name,,,,,,,,,,,,,,,,,,,,,
Aatrox,1.000000,0.000000,0.577350,0.000000,0.333333,0.288675,0.000000,0.000000,0.000000,0.00000,...,0.577350,0.666667,0.000000,0.516398,0.288675,0.000000,0.000000,0.000000,0.000000,0.000000
Ahri,0.000000,1.000000,0.288675,1.000000,0.333333,0.288675,1.000000,1.000000,0.666667,0.57735,...,0.288675,0.333333,0.666667,0.000000,0.288675,0.666667,0.866025,0.666667,1.000000,0.666667
Akali,0.577350,0.288675,1.000000,0.288675,0.288675,0.250000,0.288675,0.288675,0.000000,0.00000,...,0.750000,0.577350,0.000000,0.447214,0.750000,0.000000,0.250000,0.000000,0.288675,0.000000
Akshan,0.000000,1.000000,0.288675,1.000000,0.333333,0.288675,1.000000,1.000000,0.666667,0.57735,...,0.288675,0.333333,0.666667,0.000000,0.288675,0.666667,0.866025,0.666667,1.000000,0.666667
Alistar,0.333333,0.333333,0.288675,0.333333,1.000000,0.866025,0.333333,0.333333,0.333333,0.57735,...,0.288675,0.666667,0.666667,0.516398,0.288675,0.333333,0.288675,0.666667,0.333333,0.666667


In [85]:
df_v2.columns = df_v2.columns.str.lower()

In [93]:
def recommend_character(name, df_sim, df_v2, top_k=5):
    name = name.strip()
    if name not in df_sim.index:
        raise ValueError(f"Personagem '{name}' não encontrado.")
    sim_scores = df_sim.loc[name]
    top_sim = sim_scores.drop(name).sort_values(ascending=False).head(top_k)
    recomendacao = df_v2[df_v2['name'].isin(top_sim.index)].copy()
    recomendacao.insert(1, 'similaridade', top_sim.values)
    return recomendacao.reset_index(drop=True)


In [126]:
personagem_input = input("Digite o nome do personagem: ")
quantidade_similaridades = int(input("Escreva quantos personagens similares você quer ver: "))

top5 = recommend_character(personagem_input, df_sim, df_v2, top_k=quantidade_similaridades)
top5.head(quantidade_similaridades)


Digite o nome do personagem: Leona
Escreva quantos personagens similares você quer ver: 30


,name,similaridade,resourse type_blood well,resourse type_courage,resourse type_crimson rush,resourse type_energy,resourse type_ferocity,resourse type_flow,resourse type_fury,resourse type_grit,...,resourse type_nenhum,resourse type_rage,resourse type_shield,range type_melee,range type_ranged,bottom,jungle,middle,support,top
0,Alistar,1.000000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
1,Amumu,1.000000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,1,0
2,Bard,1.000000,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
3,Blitzcrank,1.000000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
4,Braum,1.000000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
5,Camille,1.000000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,1
6,Cho'Gath,1.000000,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
7,Darius,0.866025,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
8,Evelynn,0.866025,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0
9,Fiora,0.866025,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
